# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 04c: Customer Activity Snapshot (Schedulable Data Transformation)

---

### What This Notebook Does
Creates a daily snapshot of customer activity metrics - engagement level, channel preferences, spending patterns. Used by marketing for outreach targeting and by ops for dormant account detection.

**Output Table:** `CUSTOMER_ACTIVITY_SNAPSHOT`

> **Warehouse:** `HOL_MEDIUM_WH`

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()

## Step 1: Build Customer Activity Metrics

Compute engagement signals from transactions (last 30 days), preferred channels, and spending categories.

In [ ]:
# =============================================================================
# CUSTOMER ACTIVITY SNAPSHOT
# =============================================================================

customers_df = session.table("CUSTOMERS")
accounts_df = session.table("ACCOUNTS")
transactions_df = session.table("TRANSACTIONS")

# Last 30 days transactions joined to customer via account
recent_txns = transactions_df.filter(
    F.to_timestamp(F.col("TXN_DATE")) >= F.dateadd("day", F.lit(-30), F.current_timestamp())
).join(
    accounts_df.select("ACCOUNT_ID", "CUSTOMER_ID"),
    "ACCOUNT_ID"
)

# Aggregate per customer
activity_metrics = recent_txns.group_by("CUSTOMER_ID").agg(
    F.count("*").alias("TXN_COUNT_30D"),
    F.sum("AMOUNT").alias("TOTAL_SPEND_30D"),
    F.avg("AMOUNT").alias("AVG_TXN_AMOUNT"),
    F.max(F.to_timestamp(F.col("TXN_DATE"))).alias("LAST_TXN_DATE"),
    F.count_distinct("CHANNEL").alias("CHANNELS_USED"),
    F.count_distinct("CATEGORY").alias("CATEGORIES_USED"),
    F.count_distinct("MERCHANT").alias("UNIQUE_MERCHANTS"),
    # Most used channel (mode approximation)
    F.mode(F.col("CHANNEL")).alias("PREFERRED_CHANNEL"),
    # Most spent category
    F.mode(F.col("CATEGORY")).alias("TOP_CATEGORY")
)

# Join with customer info and classify engagement
snapshot_df = customers_df.select(
    "CUSTOMER_ID", "FIRST_NAME", "LAST_NAME", "SEGMENT", "IS_ACTIVE"
).join(activity_metrics, "CUSTOMER_ID", "left").with_columns(
    ["ENGAGEMENT_LEVEL", "DAYS_SINCE_LAST_TXN", "SNAPSHOT_DATE"],
    [
        F.when(F.col("TXN_COUNT_30D") >= F.lit(20), F.lit("HIGHLY_ACTIVE"))
         .when(F.col("TXN_COUNT_30D") >= F.lit(10), F.lit("ACTIVE"))
         .when(F.col("TXN_COUNT_30D") >= F.lit(3), F.lit("MODERATE"))
         .when(F.col("TXN_COUNT_30D") >= F.lit(1), F.lit("LOW"))
         .otherwise(F.lit("DORMANT")),
        F.datediff("day", F.col("LAST_TXN_DATE"), F.current_timestamp()),
        F.current_date()
    ]
)

print(f"Customer activity snapshot: {snapshot_df.count()} customers")
snapshot_df.group_by("ENGAGEMENT_LEVEL").count().sort("COUNT", ascending=False).show()

## Step 2: Write Snapshot

In [ ]:
# Write daily snapshot
snapshot_df.write.mode("overwrite").save_as_table("CUSTOMER_ACTIVITY_SNAPSHOT")

result = session.table("CUSTOMER_ACTIVITY_SNAPSHOT")
print(f"✅ CUSTOMER_ACTIVITY_SNAPSHOT: {result.count()} customers")
result.group_by("ENGAGEMENT_LEVEL", "SEGMENT").count().sort("ENGAGEMENT_LEVEL", "SEGMENT").show(10)